# Project 18: YOLOv8 Training & Evaluation on Raw Dark Images (Baseline)

### Research Objective:
> **Goal:** Train (fine-tune) **YOLOv8** directly on the raw, unenhanced low-light image dataset (**ExDark Dataset**) using pre-trained weights to establish the primary **Ground Baseline**.  
> **Scientific Purpose:** Serves as the fundamental benchmark to quantitatively compare against subsequent enhancement pipelines:
> 1. Benchmark 1: **Raw Dark Baseline** (This notebook)
> 2. Benchmark 2: **Classical DIP Baseline (CLAHE + Bilateral Filter)**
> 3. Benchmark 3: **Zero-DCE Cascaded (Zero-shot evaluation)**
> 4. Benchmark 4: **Zero-DCE Retrained & Aligned (Fine-tuning on enhanced images)**

---

### Configured Hyperparameters:
* `MODEL_NAME = "yolov8n.pt"`
* `EPOCHS = 40`
* `IMAGE_SIZE = 640`
* `BATCH_SIZE = 16`
* `PATIENCE = 15` (Early stopping if validation mAP does not improve for 15 consecutive epochs)
* `SEED = 42` (Fixed random seed for reproducibility)

---

### Hardware Requirements & Estimated Runtime:
* **Device:** CUDA-enabled GPU recommended (NVIDIA GPU).
* **Estimated Training Time:** ~25 - 35 minutes for 40 epochs on an NVIDIA GPU.


In [ ]:
# ==============================================================================
# STEP 1: HARDWARE GPU CHECK & REPRODUCIBILITY SETUP
# ==============================================================================
# Verify GPU availability and fix random seed for reproducible results.

import os
import sys
import random
import numpy as np
import torch

# 1. Check GPU status via nvidia-smi
print("Checking GPU hardware status...")
!nvidia-smi

# 2. Verify CUDA device via PyTorch
gpu_available = torch.cuda.is_available()
print("\n" + "="*50)
print(f"CUDA Available: {gpu_available}")
if gpu_available:
    device_count = torch.cuda.device_count()
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU Count:       {device_count}")
    print(f"Device 0 Name:   {device_name}")
    print(f"PyTorch Version: {torch.__version__}")
else:
    print("WARNING: CUDA GPU not found. Training will proceed on CPU.")
print("="*50)

# 3. Fix random seed: SEED = 42
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if gpu_available:
    torch.cuda.manual_seed_all(SEED)
print(f"Fixed Random Seed = {SEED}")


In [ ]:
# ==============================================================================
# STEP 2: INSTALL & IMPORT REQUIRED DEPENDENCIES
# ==============================================================================
# Install Ultralytics YOLOv8, OpenCV, and data visualization libraries.

print("Checking and installing dependencies...")
!pip install -q ultralytics opencv-python matplotlib seaborn pandas pyyaml tqdm Pillow

import cv2
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"OpenCV version:      {cv2.__version__}")
print("Environment setup complete.")


In [ ]:
# ==============================================================================
# STEP 3: WORKSPACE DIRECTORY & DATASET CONFIGURATION
# ==============================================================================
# Resolve project root path and generate absolute-path data_baseline.yaml.

import os
import shutil
from pathlib import Path

current_dir = Path.cwd()
print(f"Current Working Directory (CWD): {current_dir}")

PROJECT_ROOT = None
possible_paths = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    current_dir / "Low-Light-Image-Enhancement-and-Downstream-Recognition",
    Path("Low-Light-Image-Enhancement-and-Downstream-Recognition")
]

for p in possible_paths:
    if (p / "Dataset" / "exdark_yolo_dark").exists():
        PROJECT_ROOT = p.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Dataset/exdark_yolo_dark directory not found in workspace hierarchy.")

print(f"Project Root resolved: {PROJECT_ROOT}")

DATASET_DIR = (PROJECT_ROOT / "Dataset" / "exdark_yolo_dark").resolve()
assert DATASET_DIR.exists(), f"Dataset directory missing at: {DATASET_DIR}"
print(f"Dataset directory: {DATASET_DIR}")

# 12 Standard ExDark Classes
EXDARK_CLASSES = [
    'Bicycle', 'Boat', 'Bottle', 'Bus', 'Cat', 'Cup',
    'Motorbike', 'People', 'Table', 'car', 'chair', 'dog'
]

# Generate YAML configuration with absolute path
yaml_config = {
    'path': str(DATASET_DIR).replace('\\', '/'),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(EXDARK_CLASSES),
    'names': {i: name for i, name in enumerate(EXDARK_CLASSES)}
}

BASELINE_YAML_PATH = (PROJECT_ROOT / "data_baseline.yaml").resolve()
with open(BASELINE_YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_config, f, sort_keys=False, default_flow_style=False)

print(f"Configuration saved to: {BASELINE_YAML_PATH}")
with open(BASELINE_YAML_PATH, 'r', encoding='utf-8') as f:
    print("\n--- DATA_BASELINE.YAML CONTENT ---")
    print(f.read())


In [ ]:
# ==============================================================================
# STEP 4: DATASET EXPLORATION & GROUND TRUTH VALIDATION
# ==============================================================================
# Count dataset split distribution and inspect raw dark bounding boxes.

def count_files(dir_path):
    exts = ('.jpg', '.jpeg', '.png', '.bmp')
    return len([f for f in dir_path.glob('*') if f.suffix.lower() in exts])

train_img_dir = DATASET_DIR / "train" / "images"
valid_img_dir = DATASET_DIR / "valid" / "images"
test_img_dir = DATASET_DIR / "test" / "images"

n_train = count_files(train_img_dir)
n_val = count_files(valid_img_dir)
n_test = count_files(test_img_dir)
total_images = n_train + n_val + n_test

print("EXDARK DATASET STATISTICS:")
print(f"   - Training split:   {n_train} images ({n_train/total_images*100:.1f}%)")
print(f"   - Validation split: {n_val} images ({n_val/total_images*100:.1f}%)")
print(f"   - Testing split:    {n_test} images ({n_test/total_images*100:.1f}%)")
print(f"   - Total images:     {total_images}")
print(f"   - Class count:      {len(EXDARK_CLASSES)} classes")

def visualize_sample(image_path, label_path, class_names):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if label_path.exists():
        with open(label_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)
                
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label_text = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                cv2.putText(img, label_text, (x1, max(y1 - 8, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img

sample_images = list(train_img_dir.glob("*.jpg"))[:4]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, img_p in enumerate(sample_images):
    lbl_p = DATASET_DIR / "train" / "labels" / f"{img_p.stem}.txt"
    rendered = visualize_sample(img_p, lbl_p, EXDARK_CLASSES)
    if rendered is not None:
        axes[i].imshow(rendered)
    axes[i].set_title(f"Sample {i+1}: {img_p.name}", fontsize=11)
    axes[i].axis('off')

plt.suptitle("Raw Dark Images with Ground Truth Bounding Boxes (ExDark)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# STEP 5: INITIALIZE YOLOV8 WITH PRE-TRAINED WEIGHTS
# ==============================================================================
# Using Transfer Learning from MS COCO weights for faster convergence.

MODEL_NAME = "yolov8n.pt"

print(f"Loading YOLOv8 model architecture from: {MODEL_NAME}...")
model = YOLO(MODEL_NAME)

print("\n" + "="*50)
print("MODEL ARCHITECTURE SUMMARY:")
model.info()
print("="*50)
print("Model initialized and ready for training.")


In [ ]:
# ==============================================================================
# STEP 6: CONFIGURE HYPERPARAMETERS & EXECUTE FINE-TUNING
# ==============================================================================
MODEL_NAME = "yolov8n.pt"
EPOCHS = 40
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15
SEED = 42

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
OUTPUT_PROJECT = str(PROJECT_ROOT / "runs" / "detect")
RUN_NAME = "raw_dark_baseline_yolov8n"

print("STARTING YOLOV8 FINE-TUNING ON RAW DARK IMAGES...")
print(f"   - Model Name:     {MODEL_NAME}")
print(f"   - Epochs:         {EPOCHS}")
print(f"   - Image Size:     {IMAGE_SIZE}")
print(f"   - Batch Size:     {BATCH_SIZE}")
print(f"   - Early Stopping: {PATIENCE} epochs patience")
print(f"   - Random Seed:    {SEED}")
print(f"   - Device:         {DEVICE}")
print(f"   - Data Config:    {BASELINE_YAML_PATH}")
print("-" * 65)

train_results = model.train(
    data=str(BASELINE_YAML_PATH),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    patience=PATIENCE,
    device=DEVICE,
    project=OUTPUT_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    save_period=-1,
    plots=True,
    verbose=True,
    seed=SEED
)

print("\nTRAINING COMPLETE.")
train_dir = Path(train_results.save_dir)
best_weight_path = train_dir / "weights" / "best.pt"
last_weight_path = train_dir / "weights" / "last.pt"
print(f"Training run directory: {train_dir}")
print(f"Best checkpoint:        {best_weight_path}")
print(f"Last checkpoint:        {last_weight_path}")


In [ ]:
# ==============================================================================
# STEP 7: INDEPENDENT EVALUATION ON TEST SPLIT
# ==============================================================================
# Evaluate best.pt checkpoint on unseen test images.

print("Loading best checkpoint (best.pt) for test split evaluation...")
best_model = YOLO(str(best_weight_path))

test_eval = best_model.val(
    data=str(BASELINE_YAML_PATH),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(PROJECT_ROOT / "Results" / "LQA"),
    name="test_raw",
    exist_ok=True
)

precision = float(test_eval.box.mp)
recall = float(test_eval.box.mr)
map50 = float(test_eval.box.map50)
map50_95 = float(test_eval.box.map)

print("\n" + "="*60)
print("QUANTITATIVE EVALUATION SUMMARY (RAW DARK BASELINE):")
print("="*60)
print(f"   - Mean Precision (P): {precision:.4f} ({precision*100:.2f}%)")
print(f"   - Mean Recall (R):    {recall:.4f} ({recall*100:.2f}%)")
print(f"   - mAP@0.5:            {map50:.4f} ({map50*100:.2f}%)")
print(f"   - mAP@0.5:0.95:       {map50_95:.4f} ({map50_95*100:.2f}%)")
print("="*60)

class_metrics = []
for i, name in enumerate(EXDARK_CLASSES):
    ap50 = float(test_eval.box.ap50[i]) if i < len(test_eval.box.ap50) else 0.0
    ap = float(test_eval.box.ap[i]) if i < len(test_eval.box.ap) else 0.0
    class_metrics.append({
        'Class ID': i,
        'Class Name': name,
        'AP@0.5': f"{ap50:.4f}",
        'AP@0.5:0.95': f"{ap:.4f}"
    })

df_classes = pd.DataFrame(class_metrics)
print("\nPER-CLASS ACCURACY BREAKDOWN:")
print(df_classes.to_string(index=False))


In [ ]:
# ==============================================================================
# STEP 8: PLOT & VISUALIZE EXPERIMENTAL DIAGRAMS
# ==============================================================================
# Visualizing training loss curves, confusion matrices, and PR curves.

eval_dir = Path(test_eval.save_dir)

def show_image_if_exists(img_path, title, figsize=(12, 8)):
    if Path(img_path).exists():
        plt.figure(figsize=figsize)
        plt.imshow(Image.open(img_path))
        plt.title(title, fontsize=13, fontweight='bold')
        plt.axis('off')
        plt.show()

# 1. Training loss & metrics progression
print("1. Training Progression Curves (results.png):")
show_image_if_exists(train_dir / "results.png", "YOLOv8 Training Loss & Performance Metrics", figsize=(16, 8))

# 2. Confusion Matrices
print("2. Confusion Matrix Visualizations:")
show_image_if_exists(eval_dir / "confusion_matrix_normalized.png", "Normalized Confusion Matrix (Test Split)", figsize=(12, 10))
show_image_if_exists(eval_dir / "confusion_matrix.png", "Raw Confusion Matrix (Test Split)", figsize=(12, 10))

# 3. Detection performance curves (PR, F1, P, R)
print("3. Performance Evaluation Curves (PR, F1, P, R):")
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
curve_names = [
    (eval_dir / "BoxPR_curve.png", "Precision - Recall Curve (PR Curve)", 0, 0),
    (eval_dir / "BoxF1_curve.png", "F1 Score vs Confidence Curve", 0, 1),
    (eval_dir / "BoxP_curve.png", "Precision vs Confidence Curve", 1, 0),
    (eval_dir / "BoxR_curve.png", "Recall vs Confidence Curve", 1, 1),
]

for p, title, r, c in curve_names:
    ax = axes[r, c]
    if p.exists():
        ax.imshow(Image.open(p))
        ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle("Evaluation Curves on Test Split (Results/ninh Standard)", fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# 4. Qualitative detection predictions
print("4. Qualitative Prediction Batches (val_batch*_pred.jpg):")
pred_batches = list(eval_dir.glob("val_batch*_pred.jpg"))
for pb in pred_batches[:2]:
    show_image_if_exists(pb, f"Test Split Inference: {pb.name}", figsize=(16, 10))


In [ ]:
# ==============================================================================
# STEP 9: EXPORT METRICS & RUNTIME ENVIRONMENT LOGS
# ==============================================================================
import subprocess
import json
import csv

test_raw_dir = Path(test_eval.save_dir)

# 1. Export measured_metrics.json and CSV for test_raw
measured_metrics = {
    "weights": str(best_weight_path),
    "data": str(BASELINE_YAML_PATH),
    "split": "test",
    "precision": float(precision),
    "recall": float(recall),
    "map50": float(map50),
    "map50_95": float(map50_95)
}

with open(test_raw_dir / "measured_metrics.json", "w", encoding="utf-8") as f:
    json.dump(measured_metrics, f, indent=2)

with open(test_raw_dir / "measured_metrics.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(measured_metrics.keys()))
    writer.writeheader()
    writer.writerow(measured_metrics)

print(f"Exported measured_metrics to: {test_raw_dir}")

# 2. Export environment.json and environment-freeze.txt for dark_e40
env_dict = {
    "python": sys.version,
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "device": str(DEVICE),
    "data": str(BASELINE_YAML_PATH),
    "seed": SEED
}
with open(train_dir / "environment.json", "w", encoding="utf-8") as f:
    json.dump(env_dict, f, indent=2)

try:
    freeze_res = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
    if freeze_res.returncode == 0:
        (train_dir / "environment-freeze.txt").write_text(freeze_res.stdout, encoding="utf-8")
except Exception:
    pass

print(f"Exported environment logs to: {train_dir}")


In [ ]:
# ==============================================================================
# STEP 10: PACKAGE ARTIFACTS (BOTH BEST.PT & LAST.PT) INTO ZIP ARCHIVE
# ==============================================================================
# Organizes all artifacts matching Results/ninh structure:
# Results/LQA/
# ├── dark_e40/        (train curves, batches, results.csv, weights: best.pt & last.pt)
# └── test_raw/        (test curves, prediction batches, measured_metrics.json & csv)

import zipfile
import shutil

LQA_ROOT = PROJECT_ROOT / "Results" / "LQA"
LQA_DARK_E40 = LQA_ROOT / "dark_e40"
LQA_DARK_E40.mkdir(parents=True, exist_ok=True)
(LQA_DARK_E40 / "weights").mkdir(parents=True, exist_ok=True)

# 1. Copy all training artifacts to dark_e40
if 'train_dir' in globals() and train_dir.exists():
    for item in train_dir.iterdir():
        dest = LQA_DARK_E40 / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)

# 2. Ensure BOTH best.pt AND last.pt are preserved in weights/
if 'best_weight_path' in globals() and best_weight_path.exists():
    shutil.copy2(best_weight_path, LQA_DARK_E40 / "weights" / "best.pt")
    print(f"Included best weight: {best_weight_path.name} ({best_weight_path.stat().st_size / 1024 / 1024:.2f} MB)")

if 'last_weight_path' in globals() and last_weight_path.exists():
    shutil.copy2(last_weight_path, LQA_DARK_E40 / "weights" / "last.pt")
    print(f"Included last weight: {last_weight_path.name} ({last_weight_path.stat().st_size / 1024 / 1024:.2f} MB)")

# 3. Copy test evaluation artifacts to test_raw
if 'test_eval' in globals():
    eval_dir = Path(test_eval.save_dir)
    LQA_TEST_RAW = LQA_ROOT / "test_raw"
    LQA_TEST_RAW.mkdir(parents=True, exist_ok=True)
    for item in eval_dir.iterdir():
        dest = LQA_TEST_RAW / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)

# 4. Compress Results/LQA into LQA_results.zip
ZIP_OUTPUT = PROJECT_ROOT / "Results" / "LQA_results.zip"

print(f"\nCompressing full LQA results to: {ZIP_OUTPUT.name}...")
with zipfile.ZipFile(ZIP_OUTPUT, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(LQA_ROOT):
        for file in files:
            file_path = Path(root) / file
            if file_path.suffix == ".zip":
                continue
            arcname = file_path.relative_to(LQA_ROOT.parent)
            zipf.write(file_path, arcname=str(arcname).replace('\\', '/'))

print("\n" + "="*70)
print(f"SUCCESS! Archive created at: {ZIP_OUTPUT}")
print(f"   Archive size: {ZIP_OUTPUT.stat().st_size / 1024 / 1024:.2f} MB")
print("   Weights included: best.pt and last.pt")
print("="*70)

# Optional: Direct download links if running in an interactive Jupyter environment
try:
    from IPython.display import FileLink, display
    print("\nDownload Links (Click below to download):")
    display(FileLink(str(ZIP_OUTPUT)))
    if (LQA_DARK_E40 / "weights" / "best.pt").exists():
        display(FileLink(str(LQA_DARK_E40 / "weights" / "best.pt")))
    if (LQA_DARK_E40 / "weights" / "last.pt").exists():
        display(FileLink(str(LQA_DARK_E40 / "weights" / "last.pt")))
except Exception:
    pass
